# Supplementary Notebook: Cross-dataset validation of interpretability regimes

This notebook contains two complementary validation analyses referenced in the manuscript:
1. IFN-focused validation analysis
2. Multicellular melanoma robustness analysis

Code and outputs are preserved. Informal notes and exploratory commentary have been removed.


## IFN analysis

In [1]:
from pathlib import Path
import json
import random
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
from scipy.io import mmread
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA

import torch
import torch.nn as nn

warnings.filterwarnings("ignore")

def require(condition, message):
    if not condition:
        raise ValueError(message)

def set_seed(seed=7):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(7)
print("Imports ready.")

Imports ready.


## Configure paths and locked hyperparameters

In [2]:
PROJECT_ROOT = Path("/Users/sally/Desktop/Dataset_extension")

DATASET_NAME = "GSE96583_IFNB"
OUTPUT_DIR = PROJECT_ROOT / "locked_runs" / DATASET_NAME

GENE_UNIVERSE_PATH = PROJECT_ROOT / "resources" / "gene_universe.tsv"
PRUNED_REGULONS_PATH = PROJECT_ROOT / "resources" / "tf_regulons_pruned.tsv"

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "GSE96583"
CTRL_MTX = DATA_DIR / "GSM2560248_2.1.mtx.gz"
STIM_MTX = DATA_DIR / "GSM2560249_2.2.mtx.gz"
CTRL_BARCODES = DATA_DIR / "GSM2560248_barcodes.tsv.gz"
STIM_BARCODES = DATA_DIR / "GSM2560249_barcodes.tsv.gz"
GENES_FILE = DATA_DIR / "GSE96583_batch2.genes.tsv.gz"

TARGET_COLUMN = "condition"
GROUP_COLUMN = "condition"

SEED = 7
TEST_SIZE = 0.15
VAL_SIZE_WITHIN_TEMP = 0.50
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
N_EPOCHS = 60
PRINT_EVERY = 5
TOP_K = 50
MIN_TARGETS_PER_TF = 5

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
for p in [GENE_UNIVERSE_PATH, PRUNED_REGULONS_PATH, CTRL_MTX, STIM_MTX, CTRL_BARCODES, STIM_BARCODES, GENES_FILE]:
    print(p.name, p.exists())

PROJECT_ROOT: /Users/sally/Desktop/Dataset_extension
OUTPUT_DIR: /Users/sally/Desktop/Dataset_extension/locked_runs/GSE96583_IFNB
gene_universe.tsv True
tf_regulons_pruned.tsv True
GSM2560248_2.1.mtx.gz True
GSM2560249_2.2.mtx.gz True
GSM2560248_barcodes.tsv.gz True
GSM2560249_barcodes.tsv.gz True
GSE96583_batch2.genes.tsv.gz True


## Load frozen resources

In [3]:
require(GENE_UNIVERSE_PATH.exists(), f"Missing gene universe file: {GENE_UNIVERSE_PATH}")
require(PRUNED_REGULONS_PATH.exists(), f"Missing regulon file: {PRUNED_REGULONS_PATH}")

gene_universe = pd.read_csv(GENE_UNIVERSE_PATH, sep="\t")
pruned_regulons = pd.read_csv(PRUNED_REGULONS_PATH, sep="\t")

require("gene_symbol" in gene_universe.columns, "gene_universe.tsv must contain column 'gene_symbol'")
require({"tf", "target"}.issubset(pruned_regulons.columns), "tf_regulons_pruned.tsv must contain columns 'tf' and 'target'")

gene_universe["gene_symbol"] = gene_universe["gene_symbol"].astype(str)
pruned_regulons["tf"] = pruned_regulons["tf"].astype(str)
pruned_regulons["target"] = pruned_regulons["target"].astype(str)

print("Frozen genes:", gene_universe.shape[0])
print("Pruned regulon edges:", pruned_regulons.shape[0])
display(gene_universe.head())
display(pruned_regulons.head())

Frozen genes: 19222
Pruned regulon edges: 78423


,gene_symbol,gene_index
0,WASH7P,0
1,LOC729737,1
2,LOC100133331,2
3,MIR6723,3
4,LOC100288069,4


,tf,target,weight,source
0,BCL3,RPL23A,1.0,DoRothEA_OmniPath
1,BCL3,RPL27A,1.0,DoRothEA_OmniPath
2,BCL3,RPL38,1.0,DoRothEA_OmniPath
3,BCL3,PDE4DIP,1.0,DoRothEA_OmniPath
4,BCL3,CACYBP,1.0,DoRothEA_OmniPath


## Load GSE96583 batch-2 control and stimulated matrices

In [4]:
for p in [CTRL_MTX, STIM_MTX, CTRL_BARCODES, STIM_BARCODES, GENES_FILE]:
    require(p.exists(), f"Missing required file: {p}")

X_ctrl = mmread(str(CTRL_MTX)).tocsr().T
X_stim = mmread(str(STIM_MTX)).tocsr().T

barcodes_ctrl = pd.read_csv(CTRL_BARCODES, header=None, sep="\t")[0].astype(str)
barcodes_stim = pd.read_csv(STIM_BARCODES, header=None, sep="\t")[0].astype(str)

genes = pd.read_csv(GENES_FILE, header=None, sep="\t")
require(genes.shape[1] >= 2, "Expected at least two columns in batch2 gene file")
genes.columns = ["ensembl_id", "gene_symbol"] + [f"extra_{i}" for i in range(genes.shape[1] - 2)]

gene_names = genes["gene_symbol"].astype(str).copy()
missing_symbol = gene_names.isna() | (gene_names == "") | (gene_names == "nan")
gene_names.loc[missing_symbol] = genes.loc[missing_symbol, "ensembl_id"].astype(str)

print("X_ctrl shape:", X_ctrl.shape)
print("X_stim shape:", X_stim.shape)
print("barcodes ctrl:", len(barcodes_ctrl))
print("barcodes stim:", len(barcodes_stim))
print("gene names:", len(gene_names))
print("first 10 gene symbols:", gene_names.head(10).tolist())

X_ctrl shape: (14619, 35635)
X_stim shape: (14446, 35635)
barcodes ctrl: 14619
barcodes stim: 14446
gene names: 35635
first 10 gene symbols: ['MIR1302-10', 'FAM138A', 'OR4F5', 'RP11-34P13.7', 'RP11-34P13.8', 'AL627309.1', 'RP11-34P13.14', 'RP11-34P13.9', 'AP006222.2', 'RP4-669L17.10']


## Build AnnData objects, deduplicate genes, and merge

In [5]:
adata_ctrl = sc.AnnData(X_ctrl)
adata_ctrl.obs_names = ("ctrl_" + barcodes_ctrl).values
adata_ctrl.var_names = gene_names.values
adata_ctrl.obs["condition"] = "control"

adata_stim = sc.AnnData(X_stim)
adata_stim.obs_names = ("stim_" + barcodes_stim).values
adata_stim.var_names = gene_names.values
adata_stim.obs["condition"] = "stimulated"

adata_ctrl = adata_ctrl[:, ~pd.Index(adata_ctrl.var_names).duplicated(keep="first")].copy()
adata_stim = adata_stim[:, ~pd.Index(adata_stim.var_names).duplicated(keep="first")].copy()

print("ctrl shape after dedup:", adata_ctrl.shape)
print("stim shape after dedup:", adata_stim.shape)
print("ctrl var unique:", adata_ctrl.var_names.is_unique)
print("stim var unique:", adata_stim.var_names.is_unique)

adata = sc.concat([adata_ctrl, adata_stim], axis=0)

print("Combined shape:", adata.shape)
print(adata.obs["condition"].value_counts())
print("First 10 genes:", adata.var_names[:10].tolist())
print("obs_names unique:", adata.obs_names.is_unique)
print("var_names unique:", adata.var_names.is_unique)
print(
    "Gene symbols matching frozen universe:",
    sum(pd.Index(adata.var_names).isin(gene_universe["gene_symbol"].astype(str)))
)

ctrl shape after dedup: (14619, 32938)
stim shape after dedup: (14446, 32938)
ctrl var unique: True
stim var unique: True
Combined shape: (29065, 32938)
condition
control       14619
stimulated    14446
Name: count, dtype: int64
First 10 genes: ['MIR1302-10', 'FAM138A', 'OR4F5', 'RP11-34P13.7', 'RP11-34P13.8', 'AL627309.1', 'RP11-34P13.14', 'RP11-34P13.9', 'AP006222.2', 'RP4-669L17.10']
obs_names unique: True
var_names unique: True
Gene symbols matching frozen universe: 16569


## Align to the frozen gene universe

In [6]:
frozen_genes = gene_universe["gene_symbol"].astype(str).tolist()
adata.var_names = adata.var_names.astype(str)

dataset_genes = set(adata.var_names)
shared_genes = [g for g in frozen_genes if g in dataset_genes]

print("Frozen genes:", len(frozen_genes))
print("Dataset genes:", adata.n_vars)
print("Shared genes:", len(shared_genes))
require(len(shared_genes) > 5000, "Too few shared genes; check identifier mismatch.")

adata_aligned = adata[:, shared_genes].copy()

print("Aligned shape:", adata_aligned.shape)
print("First 10 aligned genes:", adata_aligned.var_names[:10].tolist())
print("Aligned var_names unique:", adata_aligned.var_names.is_unique)

Frozen genes: 19222
Dataset genes: 32938
Shared genes: 16569
Aligned shape: (29065, 16569)
First 10 aligned genes: ['FAM87B', 'LINC00115', 'FAM41C', 'SAMD11', 'NOC2L', 'KLHL17', 'PLEKHN1', 'HES4', 'ISG15', 'AGRN']
Aligned var_names unique: True


## Filter regulons to aligned genes and build the gene × TF mask

In [7]:
aligned_gene_set = set(adata_aligned.var_names)

pruned_regulons_aligned = pruned_regulons[
    pruned_regulons["target"].isin(aligned_gene_set)
].copy()

tf_counts = pruned_regulons_aligned["tf"].value_counts()
valid_tfs = tf_counts[tf_counts >= MIN_TARGETS_PER_TF].index

pruned_regulons_aligned = pruned_regulons_aligned[
    pruned_regulons_aligned["tf"].isin(valid_tfs)
].copy()

genes_order = adata_aligned.var_names.astype(str).tolist()
tfs_order = sorted(pruned_regulons_aligned["tf"].astype(str).unique().tolist())

gene_to_idx = {g: i for i, g in enumerate(genes_order)}
tf_to_idx = {tf: i for i, tf in enumerate(tfs_order)}

mask = np.zeros((len(genes_order), len(tfs_order)), dtype=np.float32)
for row in pruned_regulons_aligned[["tf", "target"]].itertuples(index=False):
    tf, target = row
    if target in gene_to_idx and tf in tf_to_idx:
        mask[gene_to_idx[target], tf_to_idx[tf]] = 1.0

print("Filtered regulon edges:", pruned_regulons_aligned.shape[0])
print("Mask shape:", mask.shape)
print("Number of TFs:", len(tfs_order))
print("Mask density:", float(mask.sum()) / mask.size)
display(pruned_regulons_aligned.head())

Filtered regulon edges: 77631
Mask shape: (16569, 315)
Number of TFs: 315
Mask density: 0.014874018893573484


,tf,target,weight,source
0,BCL3,RPL23A,1.0,DoRothEA_OmniPath
1,BCL3,RPL27A,1.0,DoRothEA_OmniPath
2,BCL3,RPL38,1.0,DoRothEA_OmniPath
3,BCL3,PDE4DIP,1.0,DoRothEA_OmniPath
4,BCL3,CACYBP,1.0,DoRothEA_OmniPath


## Encode labels and split the data

In [9]:
require(TARGET_COLUMN in adata_aligned.obs.columns, f"Missing TARGET_COLUMN in obs: {TARGET_COLUMN}")

X = adata_aligned.X.toarray().astype(np.float32)
y = adata_aligned.obs[TARGET_COLUMN].astype(str).values

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Target column:", TARGET_COLUMN)
print("Classes:", label_encoder.classes_)
print("Encoded shape:", y_encoded.shape)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_encoded,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y_encoded,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=VAL_SIZE_WITHIN_TEMP,
    random_state=SEED,
    stratify=y_temp,
)

print("X_train:", X_train.shape, "X_val:", X_val.shape, "X_test:", X_test.shape)
print("Train classes:", len(np.unique(y_train)), "Val classes:", len(np.unique(y_val)), "Test classes:", len(np.unique(y_test)))

Target column: condition
Classes: ['control' 'stimulated']
Encoded shape: (29065,)
X_train: (24705, 16569) X_val: (2180, 16569) X_test: (2180, 16569)
Train classes: 2 Val classes: 2 Test classes: 2


## Convert to tensors

In [10]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.long)
y_val_t = torch.tensor(y_val, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

mask_t = torch.tensor(mask, dtype=torch.float32)

print("X_train_t:", X_train_t.shape)
print("y_train_t:", y_train_t.shape)
print("mask_t:", mask_t.shape)

X_train_t: torch.Size([24705, 16569])
y_train_t: torch.Size([24705])
mask_t: torch.Size([16569, 315])


## Define and train MM-KPNN

In [11]:
class MMKPNN(nn.Module):
    def __init__(self, n_genes, n_concepts, n_classes, mask):
        super().__init__()
        self.mask = mask
        self.gene_to_concept = nn.Linear(n_genes, n_concepts, bias=False)
        self.concept_to_output = nn.Linear(n_concepts, n_classes)

    def forward(self, x):
        w = self.gene_to_concept.weight * self.mask.T
        z = torch.matmul(x, w.T)
        out = self.concept_to_output(z)
        return out, z

n_genes = mask_t.shape[0]
n_concepts = mask_t.shape[1]
n_classes = len(label_encoder.classes_)

model = MMKPNN(n_genes, n_concepts, n_classes, mask_t)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

print("Genes:", n_genes, "Concepts:", n_concepts, "Classes:", n_classes)

Genes: 16569 Concepts: 315 Classes: 2


In [12]:
history = []

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    optimizer.zero_grad()

    logits, _ = model(X_train_t)
    loss = criterion(logits, y_train_t)
    loss.backward()
    optimizer.step()

    if epoch % PRINT_EVERY == 0 or epoch == 1:
        model.eval()
        with torch.no_grad():
            val_logits, _ = model(X_val_t)
            val_loss = criterion(val_logits, y_val_t)

            train_acc = (logits.argmax(dim=1) == y_train_t).float().mean().item()
            val_acc = (val_logits.argmax(dim=1) == y_val_t).float().mean().item()

        history.append({
            "epoch": epoch,
            "train_loss": float(loss.item()),
            "val_loss": float(val_loss.item()),
            "train_acc": float(train_acc),
            "val_acc": float(val_acc),
        })

        print(
            f"Epoch {epoch:03d} | "
            f"Train Loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f} | "
            f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}"
        )

history_df = pd.DataFrame(history)

Epoch 001 | Train Loss: 0.7101 | Val Loss: 0.6706 | Train Acc: 0.3908 | Val Acc: 0.7174
Epoch 005 | Train Loss: 0.5757 | Val Loss: 0.5478 | Train Acc: 0.8738 | Val Acc: 0.8766
Epoch 010 | Train Loss: 0.4583 | Val Loss: 0.4358 | Train Acc: 0.8951 | Val Acc: 0.8986
Epoch 015 | Train Loss: 0.3724 | Val Loss: 0.3543 | Train Acc: 0.9104 | Val Acc: 0.9151
Epoch 020 | Train Loss: 0.3067 | Val Loss: 0.2932 | Train Acc: 0.9252 | Val Acc: 0.9284
Epoch 025 | Train Loss: 0.2548 | Val Loss: 0.2463 | Train Acc: 0.9386 | Val Acc: 0.9385
Epoch 030 | Train Loss: 0.2138 | Val Loss: 0.2097 | Train Acc: 0.9479 | Val Acc: 0.9454
Epoch 035 | Train Loss: 0.1815 | Val Loss: 0.1810 | Train Acc: 0.9550 | Val Acc: 0.9518
Epoch 040 | Train Loss: 0.1560 | Val Loss: 0.1581 | Train Acc: 0.9615 | Val Acc: 0.9578
Epoch 045 | Train Loss: 0.1358 | Val Loss: 0.1400 | Train Acc: 0.9669 | Val Acc: 0.9615
Epoch 050 | Train Loss: 0.1198 | Val Loss: 0.1262 | Train Acc: 0.9707 | Val Acc: 0.9633
Epoch 055 | Train Loss: 0.1070 |

## Extract bottleneck activations

In [13]:
model.eval()
with torch.no_grad():
    _, Z_train = model(X_train_t)
    _, Z_val = model(X_val_t)
    _, Z_test = model(X_test_t)

Z_all = torch.cat([Z_train, Z_val, Z_test], dim=0).cpu().numpy()

print("Concept activation matrix shape:", Z_all.shape)
print("First row (first 10 concepts):", Z_all[0, :10])

Concept activation matrix shape: (29065, 315)
First row (first 10 concepts): [ 0.         -0.08335363  0.10405635  0.         -0.1591425   0.14869505
 -0.16121364  0.04700436 -0.05533338  0.3809978 ]


In [14]:
row_idx = np.arange(adata_aligned.n_obs)
idx_train, idx_temp = train_test_split(
    row_idx, test_size=TEST_SIZE, random_state=SEED, stratify=y_encoded
)
idx_val, idx_test = train_test_split(
    idx_temp, test_size=VAL_SIZE_WITHIN_TEMP, random_state=SEED, stratify=y_encoded[idx_temp]
)
ordered_idx = np.concatenate([idx_train, idx_val, idx_test])

obs_aligned_ordered = adata_aligned.obs.iloc[ordered_idx].copy()
obs_names_ordered = adata_aligned.obs_names[ordered_idx]

print("Ordered obs length:", len(obs_names_ordered))
print("Matches Z rows:", len(obs_names_ordered) == Z_all.shape[0])

Ordered obs length: 29065
Matches Z rows: True


## Compute concept selectivity

In [15]:
Z_df = pd.DataFrame(Z_all, index=obs_names_ordered, columns=tfs_order)

z_max = Z_df.max(axis=1)
z_mean_others = (Z_df.sum(axis=1) - z_max) / (Z_df.shape[1] - 1)
selectivity = z_max - z_mean_others

selectivity_df = pd.DataFrame({
    "cell_id": obs_names_ordered,
    "condition": obs_aligned_ordered[TARGET_COLUMN].astype(str).values,
    "selectivity": selectivity.values,
    "top_concept": Z_df.idxmax(axis=1).values,
    "top_concept_activation": z_max.values,
})

print("Selectivity table shape:", selectivity_df.shape)
display(selectivity_df.head())
print(selectivity_df["selectivity"].describe())

Selectivity table shape: (29065, 5)


,cell_id,condition,selectivity,top_concept,top_concept_activation
0,ctrl_ATGTAAACTCCGAA-1,control,0.993241,ELK1,0.990215
1,ctrl_CAATGGACATGTGC-1,control,2.031412,STAT2,2.022659
2,ctrl_GACTACGAGAGGGT-1,control,2.710507,STAT2,2.674968
3,ctrl_TAGGTTCTAGCCAT-1,control,1.961612,STAT2,1.960360
4,ctrl_CCAGATGATCTAGG-1,control,2.077633,STAT2,2.089809


count    29065.000000
mean         6.457126
std          6.539284
min          0.456606
25%          2.443455
50%          4.403959
75%          7.077781
max         69.321915
Name: selectivity, dtype: float64


In [16]:
pca = PCA(n_components=2, random_state=SEED)
Z_pca = pca.fit_transform(Z_all)

pca_df = pd.DataFrame({
    "cell_id": obs_names_ordered,
    "condition": obs_aligned_ordered[TARGET_COLUMN].astype(str).values,
    "PC1": Z_pca[:, 0],
    "PC2": Z_pca[:, 1],
})

print("PCA shape:", Z_pca.shape)
print("Explained variance ratio:", pca.explained_variance_ratio_)
display(pca_df.head())

PCA shape: (29065, 2)
Explained variance ratio: [0.4485671  0.31736848]


,cell_id,condition,PC1,PC2
0,ctrl_ATGTAAACTCCGAA-1,control,-8.567636,1.403312
1,ctrl_CAATGGACATGTGC-1,control,-6.976724,2.036579
2,ctrl_GACTACGAGAGGGT-1,control,-3.520113,-3.894602
3,ctrl_TAGGTTCTAGCCAT-1,control,-7.080901,1.975884
4,ctrl_CCAGATGATCTAGG-1,control,-7.398341,0.940888


## Compute gene-level support

In [17]:
W = (model.gene_to_concept.weight.detach().cpu().numpy()) * mask.T
W = W.T  # concepts x genes

gene_names_aligned = adata_aligned.var_names.tolist()
tf_targets = pruned_regulons_aligned.groupby("tf")["target"].apply(set).to_dict()

support_records = []
for j, tf in enumerate(tfs_order):
    weights = W[j]
    topk_idx = np.argsort(weights)[-TOP_K:]
    topk_genes = {gene_names_aligned[i] for i in topk_idx}
    targets = tf_targets.get(tf, set())
    overlap = np.nan if len(targets) == 0 else len(topk_genes & targets) / TOP_K
    support_records.append({
        "tf": tf,
        "overlap": overlap,
        "n_targets": len(targets),
    })

support_df = pd.DataFrame(support_records)

random_overlaps = []
for tf in tfs_order:
    targets = tf_targets.get(tf, set())
    if len(targets) == 0:
        continue
    sampled = set(random.sample(gene_names_aligned, TOP_K))
    random_overlaps.append(len(sampled & targets) / TOP_K)

summary_df = pd.DataFrame({
    "metric": [
        "mean_overlap",
        "median_overlap",
        "max_overlap",
        "random_mean_overlap",
        "fraction_overlap_gt_0.05",
        "fraction_overlap_gt_0.10",
    ],
    "value": [
        float(support_df["overlap"].mean()),
        float(support_df["overlap"].median()),
        float(support_df["overlap"].max()),
        float(np.mean(random_overlaps)),
        float((support_df["overlap"] > 0.05).mean()),
        float((support_df["overlap"] > 0.10).mean()),
    ]
})

print("Support table shape:", support_df.shape)
display(support_df.head())
print("\nOverlap summary:")
print(support_df["overlap"].describe())

print("\nTop TFs by overlap:")
display(support_df.sort_values("overlap", ascending=False).head(10))

print("\nDataset-level summary:")
display(summary_df)

Support table shape: (315, 3)


,tf,overlap,n_targets
0,ABL1,0.00,10
1,ADNP,0.02,321
2,AHR,0.00,420
3,APC,0.00,12
4,ARID2,0.08,453



Overlap summary:
count    315.000000
mean       0.030984
std        0.047431
min        0.000000
25%        0.000000
50%        0.020000
75%        0.040000
max        0.660000
Name: overlap, dtype: float64

Top TFs by overlap:


,tf,overlap,n_targets
233,SMARCA4,0.66,261
270,THAP1,0.14,451
130,MAF,0.12,460
77,GFI1B,0.12,445
239,SNAI2,0.12,459
133,MAZ,0.10,462
273,TP73,0.10,456
265,TERF2,0.10,399
310,ZNF750,0.10,425
254,STAT6,0.10,324



Dataset-level summary:


,metric,value
0,mean_overlap,0.030984
1,median_overlap,0.020000
2,max_overlap,0.660000
3,random_mean_overlap,0.016508
4,fraction_overlap_gt_0.05,0.244444
5,fraction_overlap_gt_0.10,0.015873


## Save outputs

In [18]:
history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)
selectivity_df.to_csv(OUTPUT_DIR / "selectivity.csv", index=False)
pca_df.to_csv(OUTPUT_DIR / "pca_geometry.csv", index=False)
support_df.to_csv(OUTPUT_DIR / "gene_level_support.csv", index=False)
summary_df.to_csv(OUTPUT_DIR / "summary_metrics.csv", index=False)

metadata = {
    "dataset_name": DATASET_NAME,
    "project_root": str(PROJECT_ROOT),
    "data_dir": str(DATA_DIR),
    "gene_universe_path": str(GENE_UNIVERSE_PATH),
    "pruned_regulons_path": str(PRUNED_REGULONS_PATH),
    "target_column": TARGET_COLUMN,
    "group_column": GROUP_COLUMN,
    "seed": SEED,
    "test_size": TEST_SIZE,
    "val_size_within_temp": VAL_SIZE_WITHIN_TEMP,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "n_epochs": N_EPOCHS,
    "top_k": TOP_K,
    "n_cells": int(adata_aligned.n_obs),
    "n_genes": int(adata_aligned.n_vars),
    "n_tfs": int(len(tfs_order)),
    "classes": [str(x) for x in label_encoder.classes_],
}
with open(OUTPUT_DIR / "run_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved files:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print("-", p.name)

Saved files:
- gene_level_support.csv
- pca_geometry.csv
- run_metadata.json
- selectivity.csv
- summary_metrics.csv
- training_history.csv


## Multicellular analysis

## 1) Imports and helper utilities

In [1]:
# Core imports
from pathlib import Path
import json
import math
import random
import warnings

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc

from scipy import sparse
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

warnings.filterwarnings("ignore")

def require(condition, message):
    if not condition:
        raise ValueError(message)

def set_seed(seed: int = 7):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(7)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cpu


## 2) Locked configuration

Edit only the dataset-specific fields here.

In [2]:
from pathlib import Path

# === EDIT THESE FOR EACH DATASET ===
DATASET_NAME = "GSE72056_Tirosh"
TARGET_COLUMN = "condition"
GROUP_COLUMN = "condition"

PROJECT_ROOT = Path("/Users/sally/Desktop/Dataset_extension")

OUTPUT_DIR = PROJECT_ROOT / "locked_runs" / DATASET_NAME

# ✅ Correct locations (FOUND)
GENE_UNIVERSE_PATH = PROJECT_ROOT / "resources" / "gene_universe.tsv"
PRUNED_REGULONS_PATH = PROJECT_ROOT / "resources" / "tf_regulons_pruned.tsv"

# Dataset path (already correct)
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "GSE72056" / "GSE72056_melanoma_single_cell_revised_v2.txt.gz"

# Locked training hyperparameters
SEED = 7
TEST_SIZE = 0.15
VAL_SIZE_WITHIN_TEMP = 0.50
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
N_EPOCHS = 60
PRINT_EVERY = 5
TOP_K = 50

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("dataset exists:", DATA_PATH.exists())
print("gene universe exists:", GENE_UNIVERSE_PATH.exists())
print("regulons exist:", PRUNED_REGULONS_PATH.exists())
print("Output dir:", OUTPUT_DIR)

dataset exists: True
gene universe exists: True
regulons exist: True
Output dir: /Users/sally/Desktop/Dataset_extension/locked_runs/GSE72056_Tirosh


In [3]:
print("dataset exists:", DATA_PATH.exists())
print("gene universe exists:", GENE_UNIVERSE_PATH.exists())
print("regulons exist:", PRUNED_REGULONS_PATH.exists())

dataset exists: True
gene universe exists: True
regulons exist: True


## 3) Load frozen upstream artifacts

These should stay fixed across all datasets.

In [4]:
require(GENE_UNIVERSE_PATH.exists(), f"Missing gene universe file: {GENE_UNIVERSE_PATH}")
require(PRUNED_REGULONS_PATH.exists(), f"Missing regulon file: {PRUNED_REGULONS_PATH}")

gene_universe = pd.read_csv(GENE_UNIVERSE_PATH, sep="\t")
pruned_regulons = pd.read_csv(PRUNED_REGULONS_PATH, sep="\t")

require("gene_symbol" in gene_universe.columns, "gene_universe.tsv must contain column 'gene_symbol'")
require({"tf", "target"}.issubset(pruned_regulons.columns), "tf_regulons_pruned.tsv must contain columns 'tf' and 'target'")

gene_universe["gene_symbol"] = gene_universe["gene_symbol"].astype(str)
pruned_regulons["tf"] = pruned_regulons["tf"].astype(str)
pruned_regulons["target"] = pruned_regulons["target"].astype(str)

print("Frozen genes:", gene_universe.shape[0])
print("Pruned regulon edges:", pruned_regulons.shape[0])
display(gene_universe.head())
display(pruned_regulons.head())

Frozen genes: 19222
Pruned regulon edges: 78423


,gene_symbol,gene_index
0,WASH7P,0
1,LOC729737,1
2,LOC100133331,2
3,MIR6723,3
4,LOC100288069,4


,tf,target,weight,source
0,BCL3,RPL23A,1.0,DoRothEA_OmniPath
1,BCL3,RPL27A,1.0,DoRothEA_OmniPath
2,BCL3,RPL38,1.0,DoRothEA_OmniPath
3,BCL3,PDE4DIP,1.0,DoRothEA_OmniPath
4,BCL3,CACYBP,1.0,DoRothEA_OmniPath


## 5) Align the dataset to the frozen gene universe

This is the key step that keeps the analysis comparable across datasets.

In [5]:
import pandas as pd
import scanpy as sc

require(DATA_PATH.exists(), f"Missing dataset file: {DATA_PATH}")

# Load raw table
df = pd.read_csv(DATA_PATH, sep="\t", index_col=0)

print("Raw table shape:", df.shape)
print("First 10 raw row names:", df.index[:10].tolist())

# Separate metadata rows from gene-expression rows
metadata_rows = [
    "tumor",
    "malignant(1=no,2=yes,0=unresolved)",
    "non-malignant cell type (1=T,2=B,3=Macro.4=Endo.,5=CAF;6=NK)",
]

require(all(r in df.index for r in metadata_rows), "Expected Tirosh metadata rows not found")

obs = df.loc[metadata_rows].T.copy()
expr = df.drop(index=metadata_rows).copy()

# Build AnnData: cells x genes
adata = sc.AnnData(expr.T)

# Attach metadata
adata.obs["tumor"] = obs["tumor"].astype(str)
adata.obs["condition"] = obs["tumor"].astype(str)  # keep notebook convention
adata.obs["malignant_code"] = pd.to_numeric(
    obs["malignant(1=no,2=yes,0=unresolved)"], errors="coerce"
)
adata.obs["non_malignant_cell_type_code"] = pd.to_numeric(
    obs["non-malignant cell type (1=T,2=B,3=Macro.4=Endo.,5=CAF;6=NK)"], errors="coerce"
)

# Make expression numeric
adata.X = adata.X.astype(float)

print("Clean AnnData shape:", adata.shape)
print("First 5 genes:", adata.var_names[:5].tolist())
print("First 5 cells:", adata.obs_names[:5].tolist())
print("Obs columns:", adata.obs.columns.tolist())
print(adata.obs.head())

Raw table shape: (23689, 4645)
First 10 raw row names: ['tumor', 'malignant(1=no,2=yes,0=unresolved)', 'non-malignant cell type (1=T,2=B,3=Macro.4=Endo.,5=CAF;6=NK)', 'C9orf152', 'RPS11', 'ELMO2', 'CREB3L1', 'PNMA1', 'MMP2', 'TMEM216']
Clean AnnData shape: (4645, 23686)
First 5 genes: ['C9orf152', 'RPS11', 'ELMO2', 'CREB3L1', 'PNMA1']
First 5 cells: ['Cy72_CD45_H02_S758_comb', 'CY58_1_CD45_B02_S974_comb', 'Cy71_CD45_D08_S524_comb', 'Cy81_FNA_CD45_B01_S301_comb', 'Cy80_II_CD45_B07_S883_comb']
Obs columns: ['tumor', 'condition', 'malignant_code', 'non_malignant_cell_type_code']
                            tumor condition  malignant_code  \
Cy72_CD45_H02_S758_comb      72.0      72.0             1.0   
CY58_1_CD45_B02_S974_comb    58.0      58.0             1.0   
Cy71_CD45_D08_S524_comb      71.0      71.0             2.0   
Cy81_FNA_CD45_B01_S301_comb  81.0      81.0             2.0   
Cy80_II_CD45_B07_S883_comb   80.0      80.0             2.0   

                             non_malig

In [6]:
# Check duplicate gene names
var_names_series = pd.Series(adata.var_names.astype(str))
dup_mask = var_names_series.duplicated(keep=False)

print("Total genes:", adata.n_vars)
print("Duplicated gene-name entries:", dup_mask.sum())
print("Number of unique duplicated symbols:", var_names_series[dup_mask].nunique())
print("First 20 duplicated symbols:", var_names_series[dup_mask].unique()[:20].tolist())

Total genes: 23686
Duplicated gene-name entries: 4
Number of unique duplicated symbols: 2
First 20 duplicated symbols: ['MARCH1', 'MARCH2']


In [7]:
# Drop duplicated genes, keep first occurrence
var_names_series = pd.Series(adata.var_names.astype(str))
keep_mask = ~var_names_series.duplicated(keep="first").values
adata = adata[:, keep_mask].copy()

print("Shape after dropping duplicate genes:", adata.shape)
print("Are var_names unique now?", adata.var_names.is_unique)

Shape after dropping duplicate genes: (4645, 23684)
Are var_names unique now? True


In [8]:
# Align dataset genes to frozen gene universe
frozen_genes = gene_universe["gene_symbol"].astype(str).tolist()

adata.var_names = adata.var_names.astype(str)

dataset_genes = set(adata.var_names)

shared_genes = [g for g in frozen_genes if g in dataset_genes]

print("Frozen genes:", len(frozen_genes))
print("Dataset genes:", adata.n_vars)
print("Shared genes:", len(shared_genes))

require(len(shared_genes) > 5000, "Too few shared genes; check identifier mismatch.")

adata_aligned = adata[:, shared_genes].copy()

print("Aligned AnnData shape:", adata_aligned.shape)
print("First 5 aligned genes:", adata_aligned.var_names[:5].tolist())
print("Are aligned var_names unique?", adata_aligned.var_names.is_unique)

Frozen genes: 19222
Dataset genes: 23684
Shared genes: 17310
Aligned AnnData shape: (4645, 17310)
First 5 aligned genes: ['WASH7P', 'LOC729737', 'LOC100133331', 'LOC100288069', 'LINC00115']
Are aligned var_names unique? True


In [9]:
# Keep only regulon edges whose targets are in the aligned gene set
aligned_gene_set = set(adata_aligned.var_names)

pruned_regulons_aligned = pruned_regulons[
    pruned_regulons["target"].isin(aligned_gene_set)
].copy()

print("Original regulon edges:", pruned_regulons.shape[0])
print("After gene filtering:", pruned_regulons_aligned.shape[0])

# Remove TFs with too few targets (optional but usually in your pipeline)
tf_counts = pruned_regulons_aligned["tf"].value_counts()
valid_tfs = tf_counts[tf_counts >= 5].index  # keep TFs with >=5 targets

pruned_regulons_aligned = pruned_regulons_aligned[
    pruned_regulons_aligned["tf"].isin(valid_tfs)
].copy()

print("After TF filtering:", pruned_regulons_aligned.shape[0])
print("Number of TFs:", pruned_regulons_aligned["tf"].nunique())

display(pruned_regulons_aligned.head())

Original regulon edges: 78423
After gene filtering: 77272
After TF filtering: 77272
Number of TFs: 315


,tf,target,weight,source
0,BCL3,RPL23A,1.0,DoRothEA_OmniPath
1,BCL3,RPL27A,1.0,DoRothEA_OmniPath
2,BCL3,RPL38,1.0,DoRothEA_OmniPath
3,BCL3,PDE4DIP,1.0,DoRothEA_OmniPath
4,BCL3,CACYBP,1.0,DoRothEA_OmniPath


## 6) Build the fixed gene × concept mask

In [10]:
# Build gene x TF mask in aligned gene order
genes_order = adata_aligned.var_names.astype(str).tolist()
tfs_order = sorted(pruned_regulons_aligned["tf"].astype(str).unique().tolist())

gene_to_idx = {g: i for i, g in enumerate(genes_order)}
tf_to_idx = {tf: i for i, tf in enumerate(tfs_order)}

mask = np.zeros((len(genes_order), len(tfs_order)), dtype=np.float32)

for row in pruned_regulons_aligned[["tf", "target"]].itertuples(index=False):
    tf, target = row
    if target in gene_to_idx and tf in tf_to_idx:
        mask[gene_to_idx[target], tf_to_idx[tf]] = 1.0

print("Mask shape:", mask.shape)
print("Number of genes:", len(genes_order))
print("Number of TFs:", len(tfs_order))
print("Nonzero mask entries:", int(mask.sum()))
print("Mask density:", float(mask.sum()) / mask.size)

Mask shape: (17310, 315)
Number of genes: 17310
Number of TFs: 315
Nonzero mask entries: 77272
Mask density: 0.01417145791495878


In [11]:
# Encode labels from the configured target column
require(TARGET_COLUMN in adata_aligned.obs.columns, f"Missing TARGET_COLUMN in obs: {TARGET_COLUMN}")

y = adata_aligned.obs[TARGET_COLUMN].astype(str).values
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Target column:", TARGET_COLUMN)
print("Number of classes:", len(label_encoder.classes_))
print("Classes:", label_encoder.classes_[:10])
print("Encoded shape:", y_encoded.shape)

Target column: condition
Number of classes: 19
Classes: ['53.0' '58.0' '59.0' '60.0' '65.0' '67.0' '71.0' '72.0' '74.0' '75.0']
Encoded shape: (4645,)


## 7) Create reproducible train/validation/test splits

In [12]:
# Train / validation / test split
X = adata_aligned.X.astype(np.float32)

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_encoded,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y_encoded,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=VAL_SIZE_WITHIN_TEMP,
    random_state=SEED,
    stratify=y_temp,
)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)
print("Train classes:", len(np.unique(y_train)))
print("Val classes:", len(np.unique(y_val)))
print("Test classes:", len(np.unique(y_test)))

X_train: (3948, 17310) y_train: (3948,)
X_val: (348, 17310) y_val: (348,)
X_test: (349, 17310) y_test: (349,)
Train classes: 19
Val classes: 19
Test classes: 19


## 8) Build model-ready tensors

In [13]:
# Convert to torch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.long)
y_val_t = torch.tensor(y_val, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

mask_t = torch.tensor(mask, dtype=torch.float32)

print("X_train_t:", X_train_t.shape)
print("X_val_t:", X_val_t.shape)
print("X_test_t:", X_test_t.shape)
print("y_train_t:", y_train_t.shape)
print("mask_t:", mask_t.shape)

X_train_t: torch.Size([3948, 17310])
X_val_t: torch.Size([348, 17310])
X_test_t: torch.Size([349, 17310])
y_train_t: torch.Size([3948])
mask_t: torch.Size([17310, 315])


## 9) Define the fixed MM-KPNN architecture

In [14]:
# Define model
n_genes = mask_t.shape[0]
n_concepts = mask_t.shape[1]
n_classes = len(label_encoder.classes_)

class MMKPNN(nn.Module):
    def __init__(self, n_genes, n_concepts, n_classes, mask):
        super().__init__()
        self.mask = mask
        self.gene_to_concept = nn.Linear(n_genes, n_concepts, bias=False)
        self.concept_to_output = nn.Linear(n_concepts, n_classes)
        
    def forward(self, x):
        w = self.gene_to_concept.weight * self.mask.T
        z = torch.matmul(x, w.T)
        out = self.concept_to_output(z)
        return out, z

model = MMKPNN(n_genes, n_concepts, n_classes, mask_t)

print("Model initialized")
print("Genes:", n_genes)
print("Concepts:", n_concepts)
print("Classes:", n_classes)

Model initialized
Genes: 17310
Concepts: 315
Classes: 19


## 10) Train the model

This stays fixed across datasets.

In [15]:
# Training setup
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

print("Optimizer:", optimizer.__class__.__name__)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)

Optimizer: Adam
Learning rate: 0.001
Weight decay: 1e-05


In [16]:
# Training loop
for epoch in range(1, N_EPOCHS + 1):
    model.train()
    
    optimizer.zero_grad()
    logits, _ = model(X_train_t)
    loss = criterion(logits, y_train_t)
    loss.backward()
    optimizer.step()
    
    if epoch % PRINT_EVERY == 0 or epoch == 1:
        model.eval()
        with torch.no_grad():
            val_logits, _ = model(X_val_t)
            val_loss = criterion(val_logits, y_val_t)
            
            train_acc = (logits.argmax(dim=1) == y_train_t).float().mean().item()
            val_acc = (val_logits.argmax(dim=1) == y_val_t).float().mean().item()
        
        print(
            f"Epoch {epoch:03d} | "
            f"Train Loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f} | "
            f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}"
        )

Epoch 001 | Train Loss: 2.9392 | Val Loss: 2.6889 | Train Acc: 0.0904 | Val Acc: 0.1925
Epoch 005 | Train Loss: 2.4185 | Val Loss: 2.3759 | Train Acc: 0.3093 | Val Acc: 0.3046
Epoch 010 | Train Loss: 2.0721 | Val Loss: 2.0656 | Train Acc: 0.4433 | Val Acc: 0.4253
Epoch 015 | Train Loss: 1.7726 | Val Loss: 1.8004 | Train Acc: 0.5451 | Val Acc: 0.4626
Epoch 020 | Train Loss: 1.5184 | Val Loss: 1.5901 | Train Acc: 0.6087 | Val Acc: 0.5546
Epoch 025 | Train Loss: 1.2932 | Val Loss: 1.3996 | Train Acc: 0.6953 | Val Acc: 0.6322
Epoch 030 | Train Loss: 1.0942 | Val Loss: 1.2382 | Train Acc: 0.7644 | Val Acc: 0.6782
Epoch 035 | Train Loss: 0.9181 | Val Loss: 1.0967 | Train Acc: 0.8042 | Val Acc: 0.7471
Epoch 040 | Train Loss: 0.7660 | Val Loss: 0.9864 | Train Acc: 0.8483 | Val Acc: 0.7730
Epoch 045 | Train Loss: 0.6348 | Val Loss: 0.8952 | Train Acc: 0.8731 | Val Acc: 0.7960
Epoch 050 | Train Loss: 0.5223 | Val Loss: 0.8236 | Train Acc: 0.8934 | Val Acc: 0.8046
Epoch 055 | Train Loss: 0.4270 |

## 11) Extract bottleneck activations

In [17]:
# Extract concept activations (bottleneck)
model.eval()

with torch.no_grad():
    _, Z_train = model(X_train_t)
    _, Z_val = model(X_val_t)
    _, Z_test = model(X_test_t)

Z_all = torch.cat([Z_train, Z_val, Z_test], dim=0).cpu().numpy()

print("Concept activation matrix shape:", Z_all.shape)
print("First row (first 10 concepts):", Z_all[0, :10])

Concept activation matrix shape: (4645, 315)
First row (first 10 concepts): [-0.32021126  1.8801011   1.2813823   0.2519424   1.6803447   2.0236018
  0.47562632  0.05506394 -0.68052524  1.1971756 ]


## 12) Concept selectivity

This is the locked cross-dataset version: compute selectivity from the **group means** in the test set.

For each concept:
- `selectivity_score = max(group_mean) - min(group_mean)`
- `preferred_group = argmax(group_mean)`

In [18]:
# Concept selectivity
Z_df = pd.DataFrame(Z_all, index=adata_aligned.obs_names, columns=tfs_order)

z_max = Z_df.max(axis=1)
z_mean_others = (Z_df.sum(axis=1) - z_max) / (Z_df.shape[1] - 1)
selectivity = z_max - z_mean_others

selectivity_df = pd.DataFrame({
    "cell_id": adata_aligned.obs_names,
    "condition": adata_aligned.obs[TARGET_COLUMN].astype(str).values,
    "selectivity": selectivity.values,
    "top_concept": Z_df.idxmax(axis=1).values,
    "top_concept_activation": z_max.values,
})

print("Selectivity table shape:", selectivity_df.shape)
print(selectivity_df.head())
print(selectivity_df["selectivity"].describe())

Selectivity table shape: (4645, 5)
                       cell_id condition  selectivity top_concept  \
0      Cy72_CD45_H02_S758_comb      72.0     4.327654        RBPJ   
1    CY58_1_CD45_B02_S974_comb      58.0     2.616727       RUNX3   
2      Cy71_CD45_D08_S524_comb      71.0     3.885585        RBPJ   
3  Cy81_FNA_CD45_B01_S301_comb      81.0     4.791372         MAF   
4   Cy80_II_CD45_B07_S883_comb      80.0     5.153341        RBPJ   

   top_concept_activation  
0                4.305654  
1                2.556376  
2                3.799681  
3                4.758173  
4                5.135634  
count    4645.000000
mean        3.903048
std         0.964481
min         0.960243
25%         3.235384
50%         3.782999
75%         4.452739
max         8.007273
Name: selectivity, dtype: float64


In [19]:
from sklearn.decomposition import PCA

# PCA on concept activations
pca = PCA(n_components=2, random_state=SEED)
Z_pca = pca.fit_transform(Z_all)

pca_df = pd.DataFrame({
    "cell_id": adata_aligned.obs_names,
    "condition": adata_aligned.obs[TARGET_COLUMN].astype(str).values,
    "PC1": Z_pca[:, 0],
    "PC2": Z_pca[:, 1],
})

print("PCA shape:", Z_pca.shape)
print("Explained variance ratio:", pca.explained_variance_ratio_)
print(pca_df.head())

PCA shape: (4645, 2)
Explained variance ratio: [0.21577758 0.10137662]
                       cell_id condition       PC1       PC2
0      Cy72_CD45_H02_S758_comb      72.0 -2.686791 -7.374858
1    CY58_1_CD45_B02_S974_comb      58.0 -7.680264 -0.309430
2      Cy71_CD45_D08_S524_comb      71.0 -5.530390 -0.182557
3  Cy81_FNA_CD45_B01_S301_comb      81.0 -3.857416 -4.074023
4   Cy80_II_CD45_B07_S883_comb      80.0 -0.608957 -3.206741


## 14) Gene-level support

This cell keeps the same **gradient-based concept-level recovery** logic from your PBMC workflow, but makes it dataset-agnostic.

In [20]:
# Gene-level support (top-k overlap with regulon targets)

TOP_K = 50

# Get gene-to-concept weights (after masking)
W = (model.gene_to_concept.weight.detach().cpu().numpy()) * mask.T

# Transpose to concept x gene
W = W.T  # now: concepts x genes

gene_names = adata_aligned.var_names.tolist()

# Build TF → target sets
tf_targets = (
    pruned_regulons_aligned.groupby("tf")["target"]
    .apply(set)
    .to_dict()
)

support_records = []

for j, tf in enumerate(tfs_order):
    weights = W[j]  # weights for this concept
    
    # Top-k genes by absolute weight
    topk_idx = np.argsort(np.abs(weights))[-TOP_K:]
    topk_genes = {gene_names[i] for i in topk_idx}
    
    targets = tf_targets.get(tf, set())
    
    if len(targets) == 0:
        overlap = np.nan
    else:
        overlap = len(topk_genes & targets) / TOP_K
    
    support_records.append({
        "tf": tf,
        "overlap": overlap,
        "n_targets": len(targets),
    })

support_df = pd.DataFrame(support_records)

print("Support table shape:", support_df.shape)
print(support_df.head())

print("\nOverlap summary:")
print(support_df["overlap"].describe())

Support table shape: (315, 3)
      tf  overlap  n_targets
0   ABL1     0.00         10
1   ADNP     0.02        321
2    AHR     0.00        419
3    APC     0.00         12
4  ARID2     0.08        449

Overlap summary:
count    315.000000
mean       0.025397
std        0.039777
min        0.000000
25%        0.000000
50%        0.020000
75%        0.040000
max        0.520000
Name: overlap, dtype: float64


In [21]:
# Gene-level support (positive weights only)

TOP_K = 50

W = (model.gene_to_concept.weight.detach().cpu().numpy()) * mask.T
W = W.T  # concepts x genes

gene_names = adata_aligned.var_names.tolist()

tf_targets = (
    pruned_regulons_aligned.groupby("tf")["target"]
    .apply(set)
    .to_dict()
)

support_records = []

for j, tf in enumerate(tfs_order):
    weights = W[j]

    # 🔥 CHANGE: NO abs()
    topk_idx = np.argsort(weights)[-TOP_K:]
    topk_genes = {gene_names[i] for i in topk_idx}

    targets = tf_targets.get(tf, set())

    if len(targets) == 0:
        overlap = np.nan
    else:
        overlap = len(topk_genes & targets) / TOP_K

    support_records.append({
        "tf": tf,
        "overlap": overlap,
        "n_targets": len(targets),
    })

support_df_pos = pd.DataFrame(support_records)

print("Overlap summary (positive weights only):")
print(support_df_pos["overlap"].describe())

Overlap summary (positive weights only):
count    315.00000
mean       0.02546
std        0.04004
min        0.00000
25%        0.00000
50%        0.02000
75%        0.04000
max        0.52000
Name: overlap, dtype: float64


In [22]:
for TOP_K in [20, 50, 100, 200]:
    support_vals = []
    
    for j, tf in enumerate(tfs_order):
        weights = W[j]
        topk_idx = np.argsort(weights)[-TOP_K:]
        topk_genes = {gene_names[i] for i in topk_idx}
        
        targets = tf_targets.get(tf, set())
        
        if len(targets) == 0:
            continue
        
        overlap = len(topk_genes & targets) / TOP_K
        support_vals.append(overlap)
    
    support_vals = np.array(support_vals)
    
    print(f"\nTOP_K = {TOP_K}")
    print("mean:", support_vals.mean())
    print("median:", np.median(support_vals))
    print("max:", support_vals.max())


TOP_K = 20
mean: 0.03761904761904762
median: 0.0
max: 0.5

TOP_K = 50
mean: 0.025460317460317457
median: 0.02
max: 0.52

TOP_K = 100
mean: 0.024920634920634923
median: 0.02
max: 0.58

TOP_K = 200
mean: 0.021269841269841272
median: 0.015
max: 0.41


In [23]:
# Check effective regulon coverage per TF

target_sizes = []
for tf in tfs_order:
    targets = tf_targets.get(tf, set())
    aligned_targets = targets & set(gene_names)
    target_sizes.append(len(aligned_targets))

target_sizes = np.array(target_sizes)

print("Target size stats:")
print("mean:", target_sizes.mean())
print("median:", np.median(target_sizes))
print("min:", target_sizes.min())
print("max:", target_sizes.max())

print("\nFraction of TFs with < TOP_K targets:")
for k in [20, 50, 100]:
    frac = (target_sizes < k).mean()
    print(f"< {k} targets:", frac)

Target size stats:
mean: 245.30793650793652
median: 295.0
min: 9
max: 489

Fraction of TFs with < TOP_K targets:
< 20 targets: 0.17777777777777778
< 50 targets: 0.27936507936507937
< 100 targets: 0.3492063492063492


In [24]:
import random

random_overlaps = []

for j, tf in enumerate(tfs_order):
    targets = tf_targets.get(tf, set())
    
    if len(targets) == 0:
        continue
    
    random_genes = set(random.sample(gene_names, TOP_K))
    overlap = len(random_genes & targets) / TOP_K
    random_overlaps.append(overlap)

print("Random baseline:")
print("mean:", np.mean(random_overlaps))
print("median:", np.median(random_overlaps))

Random baseline:
mean: 0.014190476190476193
median: 0.01


In [25]:
support_df.sort_values("overlap", ascending=False).head(10)

,tf,overlap,n_targets
233,SMARCA4,0.52,260
270,THAP1,0.16,449
122,KLF6,0.14,448
239,SNAI2,0.14,457
28,CEBPD,0.10,471
176,NR2F1,0.10,439
113,KDM5B,0.10,458
20,BHLHE22,0.10,385
133,MAZ,0.10,461
130,MAF,0.08,459


1 TF with very high overlap → SMARCA4 (0.52)
a few moderate → ~0.10–0.16
most TFs → near 0

So the distribution is:

heavy-tailed / sparse signal

In [26]:
(support_df["overlap"] > 0.1).sum()

4

In [27]:
(support_df["overlap"] > 0.05).sum()

52

In [28]:
# fraction of useful concepts
frac_05 = (support_df["overlap"] > 0.05).mean()
frac_10 = (support_df["overlap"] > 0.1).mean()

print("Fraction > 0.05:", frac_05)
print("Fraction > 0.1:", frac_10)

Fraction > 0.05: 0.16507936507936508
Fraction > 0.1: 0.012698412698412698


mean overlap: ~0.025
random baseline: ~0.014
fraction >0.05: ~0.165
fraction >0.1: ~0.013

In [29]:
support_df.to_csv(OUTPUT_DIR / "gene_level_support.csv", index=False)
selectivity_df.to_csv(OUTPUT_DIR / "selectivity.csv", index=False)
pca_df.to_csv(OUTPUT_DIR / "pca.csv", index=False)